# Building a modified protein with mBuild can be read by OpenFF

Semaglutide is a 31-residue peptide with a lipid linker acylating one
lysine. The OpenFF post-translational-modification [workshop](https://github.com/openforcefield/2026-virtual-workshops/blob/main/ptm/ptm-workshop.ipynb) simulates
it, starting from a prepared PDB file that someone had get from somewhere.

This notebook makes that file. It starts from the unmodified peptide,
attaches the linker with mBuild, and reads the result back with
openff-pablo using the workshop's own loader call, unchanged.

## The starting structure

An ordinary protonated peptide. Residue 2 is AIB, and residue 20 is the
lysine that will carry the linker — protonated, as it is at neutral pH.

AIB is not one of the twenty standard residues, so its template is
downloaded from the RCSB on first use.

In [ ]:
from mbuild.biopolymers import Protein

mbuild_protein = Protein("semaglutide_apo.pdb", download=True)
print(mbuild_protein.n_particles, "atoms, net formal charge", mbuild_protein.net_formal_charge)

mbuild_lysine = mbuild_protein.get_residue(20, chain_id="A")
print("residue 2:", mbuild_protein.get_residue(2, chain_id="A").name)
print(f"LYS 20: charge {mbuild_lysine.formal_charge:+d}")

## The linker

The linker is a registered Chemical Component Dictionary component,
`KUT`. Building it from that component rather than from a SMILES string
matters for one reason: its atoms come out with the names the CCD gives
them. A fragment built from SMILES would get invented names, and every
tool downstream would then need to be told what those names mean.

In [ ]:
from demo_fragments import fragment_from_ccd

mbuild_linker = fragment_from_ccd("KUT", link_atom="C33")
print(mbuild_linker.n_particles, "atoms, formal charge", mbuild_linker.formal_charge)
print("bonds to the protein through:", mbuild_linker.link_atoms["1"])

## Making the bond

Two calls. `deprotonate` takes the proton off the lysine side chain,
because the ammonium ion at neutral pH has no lone pair and does not
acylate; the neutral amine does. `attach` then removes one hydrogen
from each side and forms the bond.

`leaving_atom_names` says which hydrogen goes. The three hydrogens on
that nitrogen are chemically equivalent, so the choice is arbitrary as
chemistry — but the residue library downstream describes the product by
naming the atom that is *absent*, so the file has to agree with it.

In [ ]:
mbuild_protein.deprotonate(20, "NZ", chain_id="A")
bond_record = mbuild_protein.attach(
    mbuild_linker,
    resnum=20,
    atom_name="NZ",
    chain_id="A",
    leaving_atom_names="HZ2",
    relax=False,
)

print(f"{bond_record.residue1.name} {bond_record.atom1_name} - "
      f"{bond_record.residue2.name} {bond_record.atom2_name}")
print("hydrogens removed:", bond_record.leaving1 + bond_record.leaving2)
print(mbuild_protein.n_particles, "atoms, net formal charge", mbuild_protein.net_formal_charge)

Look at the product before writing anything. `visualize` draws the
protein as a cartoon and the attached residue as licorice, from the
same PDB text that `save_pdb` writes.

In [ ]:
nglview_widget = mbuild_protein.visualize()
nglview_widget.center(selection="[KUT]")
nglview_widget

In [ ]:
mbuild_protein.save_pdb("semaglutide_mbuild.pdb", overwrite=True)
print(mbuild_protein.bond_records()[-1])

## Reading it back

This is the workshop's loader call, copied unchanged. It declares the
crosslink: which two residues, which two atoms bond, and which atom
leaves each side. Those are the same names `attach` was given.

Nothing else is needed — no hand-written residue definition, no sidecar
file. `KUT` is a CCD component, so pablo already knows it.

In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

STD_CCD_CACHE.auto_download = True
pablo_residue_library = STD_CCD_CACHE.with_crosslink(
    residues=["LYS", "KUT"],
    linking_atoms=["NZ", "C33"],
    leaving_atoms=[["HZ2"], ["H61"]],
    bond_order=1,
)

openff_topology = topology_from_pdb("semaglutide_mbuild.pdb", residue_library=pablo_residue_library)
openff_molecule = openff_topology.molecule(0)
print(openff_topology.n_molecules, "molecule:", openff_molecule.n_atoms, "atoms,",
      openff_molecule.hill_formula + ",", "net charge", openff_molecule.total_charge)

One molecule, not two. That is the whole claim: the linker came through
as part of the peptide, not as a separate thing sitting next to it.

In [ ]:
nglview_widget = openff_topology.visualize()
nglview_widget.clear_representations()
nglview_widget.add_representation("cartoon", color="#990000")
nglview_widget.add_representation("licorice", selection="[KUT]", color="#FF7733")
nglview_widget.center(selection="[KUT]")
nglview_widget

In [ ]:
openff_molecule.visualize("rdkit", show_all_hydrogens=False)

## Is it the right molecule?

`semaglutide_reference.pdb` is chain A of the workshop's own
`7KI0_prepared.pdb`. Reading both files through the same loader call
turns the question into a comparison of two OpenFF molecules.

In [ ]:
import sys

sys.path.insert(0, "scripts")
from verify_semaglutide import check

check()

Set stereochemistry aside and the two are the same molecule — not
merely the same formula, but the same graph, atom for atom and bond for
bond.

The one stereocentre that does differ is worth a word, because it is the
kind of thing this pipeline exists to surface. mBuild placed the linker
at the geometry its CCD component defines; at that centre the deposited
coordinates disagree with the component. Neither structure is malformed.
They are two different molecules, and a loader that reads chemistry
rather than coordinates is what tells you so.

## Parameters and a short simulation

From here it is the ordinary OpenFF path, the same as the workshop's.

In [ ]:
from openff.toolkit import ForceField

openff_force_field = ForceField("openff_no_water-3.0.0-alpha0.offxml", "opc3.offxml")
openff_interchange = openff_force_field.create_interchange(openff_topology)
print("parameterized", openff_interchange.topology.n_atoms, "atoms")

In [ ]:
import openmm
from openmm import unit

openmm_simulation = openff_interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picosecond, 2.0 * unit.femtosecond
    ),
)
openmm_simulation.minimizeEnergy(maxIterations=200)
openmm_simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
openmm_simulation.step(500)
openmm_state = openmm_simulation.context.getState(getEnergy=True)
print("potential energy:", openmm_state.getPotentialEnergy())

## What this needed

Reading the starting file demanded bond orders and formal charges that
are not in a PDB, so the loader matches CCD templates instead of
guessing. Writing the modified file demanded residue numbers, chain
identifiers and CONECT records for exactly the bonds a reader cannot
infer. And making the bond demanded control over which hydrogen leaves,
so that the file agrees with the residue library that reads it.

With those three, the handoff is one loader call with no special
pleading — and the structure that comes out is the one the workshop
simulates.